# CA2 – Data Visualisation and Communication
## Online Retail: Sales Analysis and Interactive Dashboard

**Student:** Carlos Eduardo Sabino Accioly Silva  
**Student number:** 2023319  
**Lecturer:** Taufique Ahmed  

**GitHub repository:** https://github.com/CarlosAccioly/ca2-online-retail-dataviz  
**Video (Google Drive):** *link to be added*

In [1]:
import pandas as pd

# Load the dataset 
df_raw = pd.read_excel("data/Online_Retail.xlsx")

print("Rows and columns:", df_raw.shape)
df_raw.head()

Rows and columns: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 1. Data Quality Assessment

Before cleaning the data, I first check its quality. This shows what problems exist and helps me decide how to fix them. I look at the structure of the data, missing values, duplicates, cancelled orders, invalid quantities and prices, and outliers.

### 1.1 Structure of the data


In [2]:
# Column names, data types and number of non-empty values
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [3]:
# Number and percentage of missing values in each column
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

pd.DataFrame({"Missing values": missing, "Missing (%)": missing_pct})

,Missing values,Missing (%)
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


**Interpretation:**

- The dataset has **541,909 rows** and **8 columns**.
- `InvoiceDate` is already stored as a date, so it can be used directly for time analysis.
- `InvoiceNo` is stored as text because some invoice numbers start with the letter "C" (cancelled orders). This is checked in a later step.
- `CustomerID` is stored as a decimal number (for example 17850.0). This happens because the column has missing values. It should be an identifier, not a number, so it will be converted during cleaning.
- `Description` has **1,454 missing values (0.27%)**. This is a very small share of the data.
- `CustomerID` has **135,080 missing values (24.93%)**. This is a large problem: one in four rows cannot be linked to a customer. These rows cannot be used for customer analysis, so how to handle them is an important cleaning decision.

### 1.2 Duplicates and cancelled orders


In [4]:
# Number of fully duplicated rows
duplicates = df_raw.duplicated().sum()
print("Duplicated rows:", duplicates)
print("Duplicated rows (%):", round(duplicates / len(df_raw) * 100, 2))

Duplicated rows: 5268
Duplicated rows (%): 0.97


In [5]:
# Cancelled orders: invoice numbers that start with "C"
cancelled = df_raw[df_raw["InvoiceNo"].astype(str).str.startswith("C")]

print("Cancelled rows:", len(cancelled))
print("Cancelled rows (%):", round(len(cancelled) / len(df_raw) * 100, 2))
cancelled.head()

Cancelled rows: 9288
Cancelled rows (%): 1.71


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


**Interpretation:**

- There are **5,268 fully duplicated rows (0.97%)**. These rows have the same invoice, product, quantity, date, price and customer. They are most likely system or data entry errors. If they are kept, revenue and quantities would be counted twice.
- There are **9,288 cancelled rows (1.71%)**. Their invoice numbers start with "C" and they have negative quantities. They are not real sales, but returns or cancelled orders.
- Some cancelled rows are not products at all. For example, stock code "D" is a **Discount**. This shows that the dataset also contains non-product entries, which are checked in the next step.

### 1.3 Invalid quantities, prices and non-product codes

In [6]:
# Rows with zero or negative quantity, and rows with zero or negative price
print("Quantity <= 0:", (df_raw["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df_raw["UnitPrice"] <= 0).sum())

Quantity <= 0: 10624
UnitPrice <= 0: 2517


In [7]:
# Stock codes that contain no digits are usually not real products
non_product = df_raw[~df_raw["StockCode"].astype(str).str.contains(r"\d")]

non_product.groupby(["StockCode", "Description"]).size().sort_values(ascending=False)

StockCode     Description               
POST          POSTAGE                       1252
DOT           DOTCOM POSTAGE                 709
M             Manual                         571
D             Discount                        77
S             SAMPLES                         63
BANK CHARGES  Bank Charges                    37
AMAZONFEE     AMAZON FEE                      34
CRUK          CRUK Commission                 16
DCGSSGIRL     GIRLS PARTY BAG                 13
DCGSSBOY      BOYS PARTY BAG                  11
PADS          PADS TO MATCH ALL CUSHIONS       4
B             Adjust bad debt                  3
m             Manual                           1
dtype: int64

**Interpretation:**

- There are **10,624 rows with zero or negative quantity**. This is more than the 9,288 cancelled rows, so some negative quantities are not cancellations. They are likely stock adjustments (for example, damaged or lost items).
- There are **2,517 rows with a price of zero or less**. A sale with no price does not create revenue, so these rows are not useful for sales analysis.
- Some stock codes are **not real products** but services or fees: postage (`POST`, `DOT`), manual adjustments (`M`), discounts (`D`), samples (`S`), bank charges, Amazon fees, CRUK commission and bad debt adjustments (`B`). These do not show customer demand for products, so they should be removed from product analysis.
- Three codes (`DCGSSGIRL`, `DCGSSBOY` and `PADS`) are real products, even though their codes have no digits. They will be kept.

### 1.4 Outliers



In [9]:
# Summary statistics for the numeric columns
df_raw[["Quantity", "UnitPrice"]].describe()


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [10]:
# Largest and smallest quantities
print("Largest quantities:")
display(df_raw.nlargest(5, "Quantity")[["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]])

print("Smallest quantities:")
display(df_raw.nsmallest(5, "Quantity")[["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID"]])

Largest quantities:


,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
540421,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0
61619,541431,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0
502122,578841,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256.0
74614,542504,NaN,5568,0.00,NaN
421632,573008,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21,12901.0


Smallest quantities:


,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
540422,C581484,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446.0
61624,C541433,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346.0
225529,556690,printing smudges/thrown away,-9600,0.00,NaN
225530,556691,printing smudges/thrown away,-9600,0.00,NaN
4287,C536757,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,15838.0


**Interpretation:**

- The data is very **skewed**. The median quantity is only 3 and the median price is £2.08, but the maximum values are much higher. The mean is also much higher than the median (9.55 vs 3 for quantity), which shows that a few extreme values pull the average up.
- The two largest orders (**80,995** and **74,215** units) were **cancelled straight away** with the same quantity (invoices C581484 and C541433). These were most likely order errors. If only the cancellation rows are removed, these large orders would stay in the data and give a false picture of sales, so both the order and its cancellation must be removed.
- Some large negative quantities have descriptions like *"printing smudges/thrown away"*, no price and no customer. These are stock adjustments, not sales.
- The minimum price (**-£11,062.06**) and the maximum price (**£38,970.00**) are far from normal product prices. They are likely accounting entries such as bad debt adjustments or manual entries. This is checked in the cleaning step.
- Not all large values are errors. A wholesaler can receive real large orders from business customers. For this reason, only outliers that are clearly invalid will be removed, and the rest will be kept.

### 1.5 Summary of data quality issues

| Issue | Rows affected | Planned action |
|---|---|---|
| Missing `CustomerID` | 135,080 (24.93%) | Keep for sales analysis, exclude from customer analysis |
| Missing `Description` | 1,454 (0.27%) | Remove |
| Duplicated rows | 5,268 (0.97%) | Remove |
| Cancelled orders (invoice starts with "C") | 9,288 (1.71%) | Remove |
| Zero or negative quantity | 10,624 | Remove |
| Zero or negative price | 2,517 | Remove |
| Non-product stock codes (postage, fees, adjustments) | Several codes | Remove |
| Extreme orders cancelled straight away | 2 orders | Remove |
| Wrong data type for `CustomerID` | Whole column | Convert to text identifier |

**Limitations:** the dataset covers only about one year (December 2010 to December 2011), so long-term trends and repeated seasons cannot be compared. It also has no information about costs, so profit cannot be measured, only revenue.